In [15]:
import pandas as pd
import sqlalchemy as sql
from datetime import date
from dateutil.relativedelta import relativedelta

In [17]:
engine = sql.create_engine('mssql+pyodbc://DESKTOP-DESBNQ8\\MSSQLSERVER01/master?driver=ODBC+driver+17+for+SQL+Server')

connect= engine.connect()

In [63]:
def extract():
    file_products = pd.read_csv("data/Products.txt", sep="|")
    db_products = pd.read_sql_query('Select * from products',con=connect)
    return file_products, db_products

def transform(file_products, db_products):
    #insert products table
    df_merge = pd.merge(file_products,db_products, how ='left', on ='product_id')
    df_insert= df_merge[df_merge['product_name_y'].isna()]
    df_insert_final = df_insert.iloc[:,0:3]
    df_insert_final.columns = db_products.columns
    #insert products_audit_scd2 table
    df_insert_final_audit= df_insert_final.copy()
    today = date.today()
    first_day_next_month = (today.replace(day=1) + relativedelta(months=1))
    df_insert_final_audit['effective_date']=first_day_next_month
    df_insert_final_audit['expire_date'] = '9999-12-31'
    df_insert_final_audit['active_flag'] = 1
    
    #update
    df_update =df_merge[((df_merge['product_name_y']!= df_merge['product_name_x']) & (df_merge['product_name_y'].notna())) | 
    ((df_merge['price_y']!= df_merge['price_x']) & (df_merge['price_y'].notna()))]
    df_update_final = df_update.iloc[:,0:3]
    df_update_final.columns = db_products.columns

    #update products_audit_scd2 table
    df_update_final_audit= df_update_final.copy()
    df_update_final_audit['effective_date']=first_day_next_month
    df_update_final_audit['expire_date'] = '9999-12-31'
    df_update_final_audit['active_flag'] = 1
    return df_insert_final, df_update_final, df_insert_final_audit, df_update_final_audit

def load(df_insert_final, df_insert_final_audit, df_update_final_audit):
    df_insert_final.to_sql('products',con=connect, index=False, if_exists ='append')
    connect.commit()
    df_insert_final_audit.to_sql('products_audit_scd2',con=connect, index=False, if_exists ='append')
    connect.commit()
    df_update_final_audit.to_sql('products_audit_scd2',con=connect, index=False, if_exists ='append')
    connect.commit()
    #update products_audit_scd2 table
    query= sql.text('''WITH cte AS (
    SELECT 
        product_key,
        product_id,
        ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY product_key desc) AS rn
    FROM products_audit_scd2
)
UPDATE p
SET active_flag = 0,
expire_date=getdate()
--Select *
FROM products_audit_scd2 p
JOIN cte c
  ON p.product_key = c.product_key
WHERE c.rn <> 1
  AND EXISTS (
      SELECT 1 
      FROM products_audit_scd2 x 
      WHERE x.product_id = p.product_id 
      HAVING COUNT(*) <> 1
  )''')
    p = connect.execute(query)
    connect.commit()

    #update products table
    query2= sql.text('''update  p set p.price = pa.price , p.product_name=pa.product_name
from Products p inner join products_audit_scd2 pa on p.product_id =pa.product_id
where pa.active_flag=1 
and (p.price<>pa.price or p.product_name<>pa.product_name)''')
    p = connect.execute(query2)
    connect.commit()


In [67]:
# Executing the ETL in DB
file_products, db_products = extract()
df_insert_final, df_update_final, df_insert_final_audit, df_update_final_audit = transform(file_products, db_products)
load(df_insert_final, df_insert_final_audit,df_update_final_audit)